# GO Enrichment Visualization — **degu background**

Identical to `scatter_GO_newlyAdded.ipynb`, except it reads the **degu-background**
enrichment results (`GO_terms_novel_genes_degubg.tsv`, produced by
`go_term_analysis_degubackground.ipynb`) and saves figures with a `_degubg` suffix.

The only other required change is the gene-set name used to select GO BP terms
(`human_GO_bp_2025.gmt`), which matches the local GMT library names from the
degu-background run.


In [ ]:
library(rrvgo)
library(org.Hs.eg.db)  # Replace with your organism DB
library(org.Mm.eg.db)  # Replace with your organism DB
library(dplyr)
library(ggplot2)
library(stringr)
library(ggrepel)
library(simplifyEnrichment)
library(stringr)


In [ ]:
go_df <- read.csv("GO_terms_novel_genes_degubg.tsv", stringsAsFactors = FALSE, sep="\t")%>%
  select(-1)  # Drops the first column

dim(go_df)

In [ ]:
go_type = "human_GO_bp_2025.gmt"
threshold = 0.05
go_df_int <- go_df[(go_df$Gene_set == go_type) & (go_df$Adjusted.P.value <= threshold), ]
dim(go_df_int)

In [ ]:
# Example GO IDs (replace with yours)
go_terms <- go_df_int$GO_ID #c("GO:0008150", "GO:0009987", "GO:0002376")

# Calculate similarity matrix (using "Rel" method)
simMatrix <- calculateSimMatrix(
    go_terms,
    orgdb = "org.Hs.eg.db",  # or org.Mm.eg.db, etc.
    ont = "BP",               # BP, MF, or CC
    method = "Rel"            # "Rel", "Wang", "Jiang"
)

In [ ]:
# Example p-values (replace with yours)
adj.pvals <- go_df_int$Adjusted.P.value	 #c(0.001, 0.01, 0.05)  

# Convert to scores (-log10(pval))
scores <- -log10(adj.pvals)
names(scores) <- go_terms  # Must match GO terms in simMatrix!


In [ ]:
reduced_terms <- reduceSimMatrix(
    simMatrix,
    scores = scores,       # Named vector (or NULL)
    threshold = 0.5,       # 0.9 (loose), 0.7 (medium), 0.5 (strict)
    orgdb = "org.Hs.eg.db" # Organism DB
)

# Preserve the original term labels before cell below overwrites reduced_terms$term
# with parentTerm (which collapses distinct GO terms onto shared cluster bars).
orig_term <- reduced_terms$term


In [ ]:
library(stringr)

wid = 10
len = 13
options(repr.plot.width = wid, repr.plot.height = len)

# Wrap the parentTerm for better display
reduced_terms$parentTerm <- str_wrap(reduced_terms$parentTerm, width = 25)

# ---- Scatter rebuilt manually so the CLUSTER labels repel each other ----
# (rrvgo::scatterPlot hardcodes geom_label_repel params; rebuilding gives control)
coords <- as.data.frame(cmdscale(as.matrix(as.dist(1 - simMatrix)), eig = TRUE, k = 2)$points)
colnames(coords) <- c("V1", "V2")
coords$go <- rownames(coords)

scat_df <- merge(coords, reduced_terms[, c("go", "term", "parent", "parentTerm", "score")], by = "go")
scat_df$is_parent <- scat_df$go == scat_df$parent
# vertical centre of the point cloud -> bias cluster labels toward the middle
v2_mid <- mean(range(scat_df$V2))
# per-label manual nudges (matched by cluster parent GO id)
scat_df$nudge_extra_x <- 0
scat_df$nudge_extra_y <- 0
scat_df$nudge_extra_x[scat_df$parent == "GO:0070098"] <- -0.25  # chemokine-mediated signaling pathway -> left white space
scat_df$nudge_extra_y[scat_df$parent == "GO:0002684"] <-  0.12  # positive regulation of immune system process -> up
# label only the 9 most significant clusters (highest -log10 adjP among their members)
cluster_sig <- tapply(scat_df$score, scat_df$parent, max)
top9_parents <- names(sort(cluster_sig, decreasing = TRUE)[1:9])
scat_df$label_this <- scat_df$is_parent & scat_df$parent %in% top9_parents

p <- ggplot(scat_df, aes(x = V1, y = V2, color = parentTerm)) +
  geom_point(aes(size = score), alpha = 0.5) +
  ggrepel::geom_label_repel(
    data = subset(scat_df, label_this),
    aes(label = parentTerm, nudge_x = nudge_extra_x, nudge_y = (v2_mid - V2) * 0.8 + nudge_extra_y),  # toward middle + per-label nudges
    size = 10,
    max.overlaps = Inf,
    min.segment.length = 0,
    box.padding = 0.6,
    point.padding = 0.6,
    label.padding = 0.35,
    force = 15,
    force_pull = 1,
    segment.color = "grey50",
    segment.size = 0.3,
    fill = "white",
    seed = 42
  ) +
  scale_color_discrete(guide = "none") +
  scale_size_continuous(guide = "none", range = c(0, 25)) +
  scale_x_continuous(name = "") +
  scale_y_continuous(name = "") +
  theme_minimal() +
  theme(axis.text.x = element_blank(), axis.text.y = element_blank())

p <- p + 
  guides(size = guide_legend(
    title = "-log10(Adj. P-value)",
    title.position = "right",
    title.hjust = 0.5
  )) +
  theme(
    legend.position = "right",
    text = element_text(size = 13),
    legend.title = element_text(
      angle = 270,
      vjust = 0.5,
      margin = margin(l = 15),
      size = 20
    ),
    panel.grid.major.y = element_blank(),
    panel.grid.major.x = element_blank(),
    legend.text = element_text(size = 18, margin = margin(l = 0.1))
  )

p

ggsave("GO_PCoAplot_degubg.svg", p, width = wid, height = len, dpi = 300)
ggsave("GO_PCoAplot_degubg.png", p, width = wid, height = len, dpi = 300)

In [ ]:
# Ensure 'go' in reduced_terms matches 'native' in go_df_int
reduced_terms_with_metadata <- reduced_terms %>% 
  left_join(
    go_df_int %>% select(GO_ID, Overlap), 
    by = c("go" = "GO_ID")  # Merge by GO ID
      )

# Ensure 'go' in reduced_terms matches 'native' in go_df_int
reduced_terms_with_metadata <- reduced_terms_with_metadata %>% 
  left_join(
    go_df_int %>% select(GO_ID, Adjusted.P.value), 
    by = c("go" = "GO_ID")  # Merge by GO ID
      )


reduced_terms_with_metadata <- reduced_terms_with_metadata %>%
  mutate(
    Overlap_fraction = sapply(strsplit(Overlap, "/"), function(x) {
      as.numeric(x[1]) / as.numeric(x[2])  # Compute numerator/denominator
    })
  )

In [ ]:
# Prepare data: calculate -log10(p-value) and order terms
go_terms <- reduced_terms_with_metadata %>%
  mutate(
    log_pvalue = -log10(Adjusted.P.value),
    term_short = gsub("\\s*\\(GO:\\d+\\)", "", orig_term)  # Remove GO IDs from labels (original term)
  ) %>%
  arrange(desc(log_pvalue))  # Sort by significance

## Add the new line if the term is way too long 
# go_terms <- go_terms %>%
#   mutate(
#     term_wrapped = sapply(term_short, function(term) {
#       if (nchar(term) > 25) {
#         # Split into words
#         words <- str_split(term, " ")[[1]]
#         current_line <- words[1]
#         result <- character(0)
        
#         # Build lines dynamically
#         for (word in words[-1]) {
#           if (nchar(paste(current_line, word)) <= 25) {
#             current_line <- paste(current_line, word)
#           } else {
#             result <- c(result, current_line)
#             current_line <- word
#           }
#         }
#         result <- c(result, current_line)
#         paste(result, collapse = "\n")
#       } else {
#         term
#       }
#     })
#   )
go_terms <- go_terms %>%
  mutate(
    term_wrapped = sapply(term_short, function(term) {
      if (nchar(term) > 25) {
        # Split into words
        words <- str_split(term, " ")[[1]]
        current_line <- words[1]
        result <- character(0)
        
        # Build lines dynamically
        for (word in words[-1]) {
          if (nchar(paste(current_line, word)) <= 25) {
            current_line <- paste(current_line, word)
          } else {
            result <- c(result, current_line)
            current_line <- word
          }
        }
        result <- c(result, current_line)
        # Add bullet point to first line, indent subsequent lines
        paste0("• ", result[1], ifelse(length(result) > 1, 
                                      paste0("\n  ", paste(result[-1], collapse = "\n  ")), 
                                      ""))
      } else {
        paste0("• ", term)
      }
    })
  )

In [ ]:
# The degu-background run yields more parent GO clusters than the original 6-color
# palette provides, so the palette is extended (original 6 colors kept first).
unique_colors = c(
  "#F8766D","#B79F00","#00BA38","#00BFC4","#619CFF","#F564E3",
  "#C77CFF","#7CAE00","#E68613","#53B400","#A58AFF","#FB61D7",
  "#00C094","#FF68A1","#DB72FB","#00B4F0","#A3A500","#FF61CC",
  "#68A180","#FA7C87","#62A6DE","#E6A00D","#A89C19","#46A34A",
  "#F04F92","#00B9B8","#D94E9E","#7B66D6","#7A7A7A","#B3531F"
)

In [ ]:
wid=17
height=15
options(repr.plot.width = wid, repr.plot.height = height)

p <- ggplot(go_terms, aes(
    x = reorder(term_wrapped, Overlap_fraction),
    y = Overlap_fraction,
    color = factor(parentTerm)  # Border color by parentTerm
  )) +
  geom_bar(
    aes(fill = -log10(Adjusted.P.value)),  # Fill by significance (viridis)
    stat = "identity",
    width = 0.7,
    linewidth = 3,
    alpha = 0.9
  ) +
  coord_flip() +
  scale_color_manual(
    name = "Parent GO term",
    values = unique_colors,
    labels = function(x) str_wrap(x, width = 20)
  ) +
  scale_fill_viridis_c(
    name = "-log10(Adj. P-value)",
    option = "plasma",
    alpha = 0.8  # Slight transparency
  ) +
  labs(
    x = "GO BP term (degu background)",
    y = "Fraction of gene set overlap",
    title = "Enriched GO terms (Adj. P-value < 0.05)"
  ) +
  theme_minimal(base_size = 12) +
  theme(
    axis.title.y = element_text(
        margin = margin(t = 10, r = 30, b = 20, l = 0, unit = "pt")
    ),
    panel.grid.major.y = element_blank(),
    # panel.grid.minor.y = element_blank(),
    legend.position = "right",
    legend.text = element_text(size = 12, margin = margin(t = 10, b = 15, unit = "pt")),
    legend.margin = margin(0, 30, 0, 0),
    legend.key.height = unit(2.2, "cm"),  # More space for colorbar
    legend.box = "vertical"  # Stack legends vertically
  ) +
  guides(
    color = guide_legend(
      override.aes = list(
        linewidth = 2,  # Thick borders in legend
        fill = "white"  # <<< White fill for legend keys
      )
    ),
    fill = guide_colorbar(
      frame.colour = "black",  # Border for colorbar
      frame.linewidth =0.5    # Thin border
    )
  )+
    theme(
            # Figure title
    plot.title = element_text(size = 22, face = "bold", hjust = 0.5),
    
    # Axis labels
    axis.title.x = element_text(size = 22, face = "bold"),
    axis.title.y = element_text(size = 22, face = "bold"),
    
    # Tick labels
    axis.text.x = element_text(size = 20, color = "black"),
    axis.text.y = element_text(size = 20, color = "black"),
    
      legend.title = element_text(size = 20),  # All legend titles
      legend.text = element_text(size = 19,    # All legend labels,
                                    margin = margin(l = 5))  # Adds 10pt space to the right of labels
    )

p
# Save as SVG
ggsave(
  filename = "GO_barplot_degubg.svg",
  plot = p,
  width = wid,
  height = height,
  dpi = 300
)

# Save as SVG
ggsave(
  filename = "GO_barplot_degubg.png",
  plot = p,
  width = wid,
  height = height,
  dpi = 300
)


In [ ]:
# ---- Companion figure: one bar per cluster, the best (most-overlap) GO term ----
best_terms <- go_terms %>%
  group_by(parentTerm) %>%
  slice_max(Overlap_fraction, n = 1, with_ties = FALSE) %>%
  ungroup()

wid=17
height=10
options(repr.plot.width = wid, repr.plot.height = height)

p_best <- ggplot(best_terms, aes(
    x = reorder(term_wrapped, Overlap_fraction),
    y = Overlap_fraction,
    color = factor(parentTerm)  # Border color by parentTerm
  )) +
  geom_bar(
    aes(fill = -log10(Adjusted.P.value)),  # Fill by significance
    stat = "identity",
    width = 0.7,
    linewidth = 3,
    alpha = 0.9
  ) +
  coord_flip() +
  scale_color_manual(
    name = "Parent GO term",
    values = unique_colors,
    labels = function(x) str_wrap(x, width = 20)
  ) +
  scale_fill_viridis_c(
    name = "-log10(Adj. P-value)",
    option = "plasma",
    alpha = 0.8
  ) +
  labs(
    x = "GO BP term (degu background)",
    y = "Fraction of gene set overlap",
    title = "Best (most-overlap) GO term per cluster"
  ) +
  theme_minimal(base_size = 12) +
  theme(
    axis.title.y = element_text(margin = margin(t = 10, r = 30, b = 20, l = 0, unit = "pt")),
    panel.grid.major.y = element_blank(),
    legend.position = "right",
    legend.text = element_text(size = 12, margin = margin(t = 10, b = 15, unit = "pt")),
    legend.margin = margin(0, 30, 0, 0),
    legend.key.height = unit(2.2, "cm"),
    legend.box = "vertical"
  ) +
  guides(
    color = guide_legend(override.aes = list(linewidth = 2, fill = "white")),
    fill = guide_colorbar(frame.colour = "black", frame.linewidth = 0.5)
  ) +
  theme(
    plot.title = element_text(size = 22, face = "bold", hjust = 0.5),
    axis.title.x = element_text(size = 22, face = "bold"),
    axis.title.y = element_text(size = 22, face = "bold"),
    axis.text.x = element_text(size = 20, color = "black"),
    axis.text.y = element_text(size = 20, color = "black"),
    legend.title = element_text(size = 20),
    legend.text = element_text(size = 19, margin = margin(l = 5))
  )

p_best
ggsave(
  filename = "GO_barplot_degubg_bestPerCluster.svg",
  plot = p_best,
  width = wid,
  height = height,
  dpi = 300
)
ggsave(
  filename = "GO_barplot_degubg_bestPerCluster.png",
  plot = p_best,
  width = wid,
  height = height,
  dpi = 300
)

In [ ]:
# ---- Companion figure: only the top 5 clusters, showing every member GO term ----
# "Top 5" = the 5 clusters whose best (most-overlap) term ranks highest, as in
# the bestPerCluster figure above. All member terms of those 5 clusters are drawn,
# each bar outlined by its parent cluster color.
best_terms <- go_terms %>%
  group_by(parentTerm) %>%
  slice_max(Overlap_fraction, n = 1, with_ties = FALSE) %>%
  ungroup()

top5_parents <- best_terms %>%
  arrange(desc(Overlap_fraction)) %>%
  head(5) %>%
  pull(parentTerm)

top5_go_terms <- go_terms %>% filter(parentTerm %in% top5_parents)

wid=18
height=17
options(repr.plot.width = wid, repr.plot.height = height)

p_top5 <- ggplot(top5_go_terms, aes(
    x = reorder(term_wrapped, Overlap_fraction),
    y = Overlap_fraction,
    color = factor(parentTerm)  # Border color by parentTerm
  )) +
  geom_bar(
    aes(fill = -log10(Adjusted.P.value)),  # Fill by significance
    stat = "identity",
    width = 0.7,
    linewidth = 3,
    alpha = 0.9
  ) +
  coord_flip() +
  scale_color_manual(
    name = "Parent GO term",
    values = unique_colors,
    labels = function(x) str_wrap(x, width = 20)
  ) +
  scale_fill_viridis_c(
    name = "-log10(Adj. P-value)",
    option = "plasma",
    alpha = 0.8
  ) +
  labs(
    x = "GO BP term (degu background)",
    y = "Fraction of gene set overlap",
    title = "Top 5 parent clusters: all member GO terms"
  ) +
  theme_minimal(base_size = 12) +
  theme(
    axis.title.y = element_text(margin = margin(t = 10, r = 30, b = 20, l = 0, unit = "pt")),
    panel.grid.major.y = element_blank(),
    legend.position = "right",
    legend.text = element_text(size = 12, margin = margin(t = 10, b = 15, unit = "pt")),
    legend.margin = margin(0, 30, 0, 0),
    legend.key.height = unit(2.2, "cm"),
    legend.box = "vertical"
  ) +
  guides(
    color = guide_legend(override.aes = list(linewidth = 2, fill = "white")),
    fill = guide_colorbar(frame.colour = "black", frame.linewidth = 0.5)
  ) +
  theme(
    plot.title = element_text(size = 22, face = "bold", hjust = 0.5),
    axis.title.x = element_text(size = 22, face = "bold"),
    axis.title.y = element_text(size = 22, face = "bold"),
    axis.text.x = element_text(size = 20, color = "black"),
    axis.text.y = element_text(size = 20, color = "black"),
    legend.title = element_text(size = 20),
    legend.text = element_text(size = 19, margin = margin(l = 5))
  )

p_top5
ggsave(
  filename = "GO_barplot_degubg_top5clusters.svg",
  plot = p_top5,
  width = wid,
  height = height,
  dpi = 300
)
ggsave(
  filename = "GO_barplot_degubg_top5clusters.png",
  plot = p_top5,
  width = wid,
  height = height,
  dpi = 300
)